# Lesson 3: Standalone Session from DataFrames

**Module:** AGA Fundamentals  
**Dataset:** Movies (synthetic — built from Pandas DataFrames in this notebook)

A *standalone* session has no AuraDB attached. You feed it Pandas DataFrames, run algorithms, and pull results back to Python. Useful when your data isn't in Neo4j yet — a CSV export, a Parquet file, anything you can land in Pandas.

We'll build a small Actor-COLLABORATED-Actor graph from two DataFrames and run PageRank on it.

## Step 1 — Setup

Same imports as before, plus `pandas` (which we'll actually use this time) and `CloudLocation` (standalone sessions need an explicit region).

In [ ]:
# Imports and environment variables
import os
import pandas as pd
from datetime import timedelta
from IPython.display import display
from graphdatascience.session import GdsSessions, AuraAPICredentials
from graphdatascience.session import SessionMemory, CloudLocation
from dotenv import load_dotenv

load_dotenv()

# Standalone sessions only need API credentials — no DBMS connection
client_id = os.getenv('AURA_CLIENT_ID')
client_secret = os.getenv('AURA_CLIENT_SECRET')

In [ ]:
# Create sessions manager
sessions = GdsSessions(
    api_credentials=AuraAPICredentials(
        client_id,
        client_secret
    )
)

## Step 2 — Create a standalone session

The key differences from an attached session:

- **No `db_connection`** — there's no AuraDB to read from.
- **`cloud_location` is required** — standalone sessions have nowhere to inherit a region from.

`sessions.available_cloud_locations()` lists the valid `(provider, region)` pairs if you're not sure which to pick.

In [ ]:
# Create a standalone GDS Session
gds = sessions.get_or_create(
    session_name="standalone-demo",
    memory=SessionMemory.m_2GB,
    ttl=timedelta(minutes=30),
    cloud_location=CloudLocation("gcp", "europe-west1")
)
print(f"Client ID starts with: {client_id[:12]}...")
gds.verify_connectivity()
print(f"Connected to GDS Session: standalone-demo")

## Step 3 — Build the DataFrames

`gds.graph.construct(...)` requires specific column shapes:

- **Nodes frame** — `nodeId` (int), `labels` (str). Extra numeric properties are fine.
- **Relationships frame** — `sourceNodeId` (int), `targetNodeId` (int), `relationshipType` (str). Extra numeric properties are fine.

**String node properties aren't supported in `construct`.** If your real data has a `name` or `title` column, drop it before calling — we'll keep actor names on the side and merge them back into the result later.

In [ ]:
# A small actor-collaboration network
actors = pd.DataFrame({
    "nodeId": [0, 1, 2, 3, 4, 5],
    "labels": ["Actor"] * 6,
    "born":   [1953, 1962, 1956, 1964, 1958, 1971],
})

# Names can't go through construct — keep them on the side
actor_names = pd.DataFrame({
    "nodeId": [0, 1, 2, 3, 4, 5],
    "name": ["Tom Hanks", "Tom Cruise", "Meryl Streep",
             "Sandra Bullock", "Kevin Bacon", "Winona Ryder"],
})

edges = pd.DataFrame({
    "sourceNodeId":     [0, 0, 1, 2, 2, 3, 4],
    "targetNodeId":     [4, 2, 3, 4, 5, 5, 5],
    "relationshipType": ["COLLABORATED"] * 7,
    "collaborations":   [3.0, 1.0, 2.0, 1.0, 4.0, 2.0, 1.0],
})

print(f"Actors: {len(actors)} | Edges: {len(edges)}")

## Step 4 — Construct the projection

One call, no Cypher. The session builds the graph directly from the two frames.

In [ ]:
G = gds.graph.construct("collaborations", actors, edges)

print(f"Projected graph: {G.name()}")
print(f"  Nodes: {G.node_count():,}")
print(f"  Relationships: {G.relationship_count():,}")

## Step 5 — Run PageRank

Same call you'd make against an attached session. In standalone mode `stream` is your only output route — there's no database for `write` to write to.

In [ ]:
df = gds.pageRank.stream(
    G,
    relationshipWeightProperty="collaborations"
)
display(df.sort_values("score", ascending=False))

## Step 6 — Recover names from the side DataFrame

The result has raw `nodeId`s. Merge them against `actor_names` to get human-readable identifiers — the standalone equivalent of `db_node_properties` in attached mode.

In [ ]:
ranked = (
    df.merge(actor_names, on="nodeId", how="left")
      .sort_values("score", ascending=False)
      [["name", "score"]]
)
display(ranked)

## Step 7 — Clean up

Same as attached sessions. Nothing in a standalone session persists once it ends, so make sure you've captured everything you want to keep from `stream` results before deletion.

In [ ]:
sessions.delete(session_name="standalone-demo")

That's it — the full standalone workflow.

You've completed Module 3:

- Authenticated against the Aura API from Python (`1_connect`).
- Run the full attached AGA workflow against AuraDB (`2_syntax`).
- Run an algorithm on data that isn't in Neo4j yet (`3_standalone`).

You're now equipped to script any AGA workflow end to end.